In [1]:
"""
Test putbarablocktriplet — variable SDP 2×2, solution analytique exacte
=======================================================================

PROBLÈME :
  minimiser  tr(X)  =  X[0,0] + X[1,1]

  sous       X[0,0] + 2·X[1,0] + X[1,1]  =  4     (contrainte 0)
             X ⪰ 0,  X ∈ S²

MATRICE DE COEFFICIENT A (triangulaire inférieure) :
  (k=0, l=0) = 1   ← diagonal
  (k=1, l=0) = 1   ← hors-diagonal  →  MOSEK calcule 2·X[1,0]
  (k=1, l=1) = 1   ← diagonal

  La contrainte devient : X[0,0] + 2·X[1,0] + X[1,1] = 4
  ce qui s'écrit        : vᵀ X v = 4   avec v = [1, 1]ᵀ

SOLUTION ANALYTIQUE :
  X* = vvᵀ = [[1, 1],   (matrice de rang 1)
               [1, 1]]

  Vérif : vᵀ X* v = 1 + 2 + 1 = 4 ✓
  tr(X*) = 2  (valeur optimale)

  Borne inférieure : tr(X) ≥ vᵀXv / ‖v‖² = 4/2 = 2  → atteinte par X*.
"""

import numpy as np
import mosek

TOL = 1e-5

def solve():
    with mosek.Task() as task:

        # Variable SDP : 1 matrice 2×2
        task.appendbarvars([2])

        # 1 contrainte scalaire
        task.appendcons(1)

        # ── Objectif : tr(X) = <I, X> ──────────────────────────────────────
        # C = I₂  →  entrées diagonales uniquement
        task.putbarcblocktriplet(
            [0, 0],       # j : indice variable SDP
            [0, 1],       # k : ligne
            [0, 1],       # l : colonne  (k == l → diagonal)
            [1.0, 1.0],   # valeur
        )

        # ── Contrainte : <A, X> = 4 ────────────────────────────────────────
        # A[0,0]=1 (diag), A[1,0]=1 (hors-diag), A[1,1]=1 (diag)
        # → produit interne = X[0,0] + 2·X[1,0] + X[1,1]
        task.putbarablocktriplet(
            [0,   0,   0  ],   # i : contrainte
            [0,   0,   0  ],   # j : variable SDP
            [0,   1,   1  ],   # k : ligne   (k >= l)
            [0,   0,   1  ],   # l : colonne
            [1.0, 1.0, 1.0],   # valeur réelle de A[k,l]
        )

        # Borne : égalité à 4
        task.putconbound(0, mosek.boundkey.fx, 4.0, 4.0)

        task.optimize()

        barx = task.getbarxj(mosek.soltype.itr, 0)
        # barx = [X[0,0], X[1,0], X[1,1]]  (ordre triangulaire inférieur)
        X = np.array([
            [barx[0], barx[1]],
            [barx[1], barx[2]],
        ])
        return X


def check(X):
    X_ref   = np.array([[1., 1.], [1., 1.]])
    opt_ref = 2.0

    print("\n── Solution MOSEK ──────────────────────")
    print(np.array2string(X, precision=6, suppress_small=True))
    print(f"\n── Valeur optimale  tr(X*) = {np.trace(X):.6f}  (attendu {opt_ref})")

    tests = {
        "Valeur optimale (tr = 2)":
            abs(np.trace(X) - opt_ref) < TOL,
        "Contrainte (X[0,0]+2·X[1,0]+X[1,1] = 4)":
            abs(X[0,0] + 2*X[1,0] + X[1,1] - 4.0) < TOL,
        "SDP (λ_min ≥ 0)":
            np.linalg.eigvalsh(X).min() >= -TOL,
        "Proximité X* analytique":
            np.linalg.norm(X - X_ref, "fro") < 1e-3,
    }

    print()
    ok = True
    for label, passed in tests.items():
        print(f"  {'✅' if passed else '❌'}  {label}")
        ok = ok and passed

    print()
    print("✅ OK" if ok else "❌ ÉCHEC — vérifiez vos valeurs hors-diagonales")


if __name__ == "__main__":
    check(solve())


── Solution MOSEK ──────────────────────
[[1. 1.]
 [1. 1.]]

── Valeur optimale  tr(X*) = 2.000000  (attendu 2.0)

  ✅  Valeur optimale (tr = 2)
  ✅  Contrainte (X[0,0]+2·X[1,0]+X[1,1] = 4)
  ✅  SDP (λ_min ≥ 0)
  ✅  Proximité X* analytique

✅ OK
